In [ ]:
# ============================================================
# DOWNLOAD E EXTRAÇÃO DAS FONTES BRONZE
# ============================================================

from __future__ import annotations

import sys
import zipfile

from pathlib import Path

import requests
import urllib3

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


urllib3.disable_warnings(
    urllib3.exceptions.InsecureRequestWarning
)


# ============================================================
# 1. LOCALIZAR A RAIZ DO PROJETO
# ============================================================

def localizar_raiz_projeto() -> Path:
    diretorio_atual = Path.cwd().resolve()

    for candidato in [
        diretorio_atual,
        *diretorio_atual.parents,
    ]:
        arquivo_secrets = (
            candidato
            / "src"
            / "config"
            / "secrets.py"
        )

        if arquivo_secrets.is_file():
            return candidato

    for candidato in [
        diretorio_atual,
        *diretorio_atual.parents,
    ]:
        possui_src = (
            candidato
            / "src"
        ).is_dir()

        possui_data = (
            candidato
            / "data"
        ).is_dir()

        if possui_src and possui_data:
            return candidato

    raise FileNotFoundError(
        "A raiz do projeto não foi encontrada. "
        "Era esperado src/config/secrets.py ou "
        "uma pasta contendo src e data."
    )


BASE_DIR = localizar_raiz_projeto()

if str(BASE_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(BASE_DIR),
    )


LOCAL_BRONZE_PATH = (
    BASE_DIR
    / "data"
    / "bronze"
)

LOCAL_BRONZE_PATH.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    f"✅ Raiz do projeto: "
    f"{BASE_DIR}"
)

print(
    f"✅ Pasta Bronze: "
    f"{LOCAL_BRONZE_PATH}"
)


# ============================================================
# 2. DOWNLOAD
# ============================================================

def download(
    url: str,
    destino: Path,
) -> Path:
    destino = Path(
        destino
    )

    destino.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    sessao = requests.Session()

    retry = Retry(
        total=10,
        backoff_factor=5,
    )

    adaptador = HTTPAdapter(
        max_retries=retry
    )

    sessao.mount(
        "https://",
        adaptador,
    )

    sessao.mount(
        "http://",
        adaptador,
    )

    with sessao.get(
        url,
        stream=True,
        verify=False,
        timeout=(
            60,
            3600,
        ),
    ) as resposta:
        resposta.raise_for_status()

        with destino.open(
            mode="wb"
        ) as arquivo_saida:
            for chunk in resposta.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    arquivo_saida.write(
                        chunk
                    )

    if not destino.is_file():
        raise RuntimeError(
            f"O arquivo não foi criado: "
            f"{destino}"
        )

    if destino.stat().st_size <= 0:
        raise RuntimeError(
            f"O arquivo está vazio: "
            f"{destino}"
        )

    print(
        f"✅ Download concluído: "
        f"{destino.name}"
    )

    return destino


# ============================================================
# 3. EXTRAÇÃO DOS ARQUIVOS ZIP
# ============================================================

def descompactar_zip(
    arquivo_zip: Path,
) -> Path:
    arquivo_zip = Path(
        arquivo_zip
    )

    if not arquivo_zip.is_file():
        raise FileNotFoundError(
            f"ZIP não encontrado: "
            f"{arquivo_zip}"
        )

    if not zipfile.is_zipfile(
        arquivo_zip
    ):
        raise zipfile.BadZipFile(
            f"Arquivo ZIP inválido: "
            f"{arquivo_zip}"
        )

    pasta_destino = (
        arquivo_zip.parent
        / arquivo_zip.stem
    )

    marcador = (
        pasta_destino
        / ".extracao_concluida"
    )

    if marcador.is_file():
        print(
            f"✅ Já descompactado: "
            f"{pasta_destino.name}"
        )

        return pasta_destino

    pasta_destino.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        f"📦 Descompactando: "
        f"{arquivo_zip.name}"
    )

    with zipfile.ZipFile(
        arquivo_zip,
        mode="r",
        allowZip64=True,
    ) as arquivo:
        arquivo.extractall(
            path=pasta_destino
        )

    arquivos_extraidos = [
        caminho
        for caminho
        in pasta_destino.rglob("*")
        if caminho.is_file()
    ]

    if not arquivos_extraidos:
        raise RuntimeError(
            f"Nenhum arquivo foi extraído de: "
            f"{arquivo_zip}"
        )

    marcador.write_text(
        (
            f"arquivo_origem={arquivo_zip.name}\n"
            f"arquivos_extraidos="
            f"{len(arquivos_extraidos)}\n"
        ),
        encoding="utf-8",
    )

    print(
        f"✅ Descompactado em: "
        f"{pasta_destino}"
    )

    print(
        f"✅ Arquivos extraídos: "
        f"{len(arquivos_extraidos):,}"
    )

    return pasta_destino


# ============================================================
# 4. FONTES
# ============================================================

ARQUIVOS = [
    {
        "nome": "resultados_ufs_2023.xlsx",
        "url": (
            "https://download.inep.gov.br/"
            "avaliacao_da_alfabetizacao/"
            "resultados_e_metas_ufs.xlsx"
        ),
    },
    {
        "nome": "resultados_municipios_2023.xlsx",
        "url": (
            "https://download.inep.gov.br/"
            "avaliacao_da_alfabetizacao/"
            "resultados_e_metas_municipios.xlsx"
        ),
    },
    {
        "nome": "microdados_2023.zip",
        "url": (
            "https://download.inep.gov.br/"
            "dados_abertos/"
            "microdados_avaliacao_da_alfabetizacao_2023.zip"
        ),
    },
    {
        "nome": "resultados_ufs_2024.xlsx",
        "url": (
            "https://download.inep.gov.br/"
            "alfabetiza_brasil/"
            "resultados_e_metas_ufs_2024_2.xlsx"
        ),
    },
    {
        "nome": "resultados_municipios_2024.xlsx",
        "url": (
            "https://download.inep.gov.br/"
            "alfabetiza_brasil/"
            "resultados_e_metas_municipios_2024.xlsx"
        ),
    },
    {
        "nome": "microdados_2024.zip",
        "url": (
            "https://download.inep.gov.br/"
            "dados_abertos/"
            "microdados_avaliacao_da_alfabetizacao_2024.zip"
        ),
    },
    {
        "nome": "resultados_ufs_2025.xlsx",
        "url": (
            "https://download.inep.gov.br/"
            "avaliacao_da_alfabetizacao/"
            "resultados/"
            "resultados_e_metas_ufs_2025_v1.xlsx"
        ),
    },
    {
        "nome": "resultados_municipios_2025.xlsx",
        "url": (
            "https://download.inep.gov.br/"
            "avaliacao_da_alfabetizacao/"
            "resultados/"
            "resultados_e_metas_municipios_2025_3.xlsx"
        ),
    },
    {
        "nome": "microdados_2025.zip",
        "url": (
            "https://download.inep.gov.br/"
            "dados_abertos/"
            "microdados_AEEB_2025.zip"
        ),
    },
    {
        "nome": "ibge_municipios.json",
        "url": (
            "https://servicodados.ibge.gov.br/"
            "api/v1/localidades/municipios"
        ),
    },
    {
        "nome": "ibge_estados.json",
        "url": (
            "https://servicodados.ibge.gov.br/"
            "api/v1/localidades/estados"
        ),
    },
    {
        "nome": "ideb_anos_iniciais.zip",
        "url": (
            "http://download.inep.gov.br/"
            "ideb/resultados/"
            "divulgacao_anos_iniciais_municipios_2025.zip"
        ),
    },
    {
        "nome": "inse_2023_municipios.xlsx",
        "url": (
            "http://download.inep.gov.br/"
            "informacoes_estatisticas/"
            "indicadores_educacionais/"
            "2023/nivel_socioeconomico/"
            "INSE_2023_municipios.xlsx"
        ),
    },
    {
        "nome": "pib_municipal.json",
        "url": (
            "https://apisidra.ibge.gov.br/"
            "values/t/5938/n6/all/"
            "v/37,516,520,528,6574/p/2021"
        ),
    },
    {
        "nome": "censo_2022.json",
        "url": (
            "https://apisidra.ibge.gov.br/"
            "values/t/4714/n6/all/"
            "v/93,6318,614/p/2022"
        ),
    },
]


# ============================================================
# 5. BAIXAR E EXTRAIR
# ============================================================

for indice, arquivo in enumerate(
    ARQUIVOS,
    start=1,
):
    nome = arquivo["nome"]

    url = arquivo["url"]

    destino = (
        LOCAL_BRONZE_PATH
        / nome
    )

    print()
    print(
        f"[{indice}/{len(ARQUIVOS)}] "
        f"{nome}"
    )

    if (
        destino.is_file()
        and destino.stat().st_size > 0
    ):
        print(
            f"✅ Já existe: "
            f"{nome}"
        )

    else:
        print(
            f"⬇️ Baixando: "
            f"{nome}"
        )

        download(
            url=url,
            destino=destino,
        )

    if destino.suffix.lower() == ".zip":
        descompactar_zip(
            destino
        )


# ============================================================
# 6. RESULTADO
# ============================================================

print()
print(
    "=" * 60
)

print(
    "✅ DOWNLOAD E EXTRAÇÃO CONCLUÍDOS"
)

print(
    "=" * 60
)

print(
    f"📁 Bronze: "
    f"{LOCAL_BRONZE_PATH}"
)

✅ Raiz do projeto: /home/linux/projetos/Teste_02112026
✅ Pasta Bronze: /home/linux/projetos/Teste_02112026/data/bronze

[1/15] resultados_ufs_2023.xlsx
⬇️ Baixando: resultados_ufs_2023.xlsx
✅ Download concluído: resultados_ufs_2023.xlsx

[2/15] resultados_municipios_2023.xlsx
⬇️ Baixando: resultados_municipios_2023.xlsx
✅ Download concluído: resultados_municipios_2023.xlsx

[3/15] microdados_2023.zip
⬇️ Baixando: microdados_2023.zip
✅ Download concluído: microdados_2023.zip
📦 Descompactando: microdados_2023.zip
✅ Descompactado em: /home/linux/projetos/Teste_02112026/data/bronze/microdados_2023
✅ Arquivos extraídos: 19

[4/15] resultados_ufs_2024.xlsx
⬇️ Baixando: resultados_ufs_2024.xlsx
✅ Download concluído: resultados_ufs_2024.xlsx

[5/15] resultados_municipios_2024.xlsx
⬇️ Baixando: resultados_municipios_2024.xlsx
✅ Download concluído: resultados_municipios_2024.xlsx

[6/15] microdados_2024.zip
⬇️ Baixando: microdados_2024.zip
✅ Download concluído: microdados_2024.zip
📦 Descompactan